## Peak number density: simulations vs. BBKS

In [ ]:
import csv

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad
from scipy.special import erf

%matplotlib inline

plt.rcParams.update({
    "figure.figsize": (7.5, 5),
    "font.size": 11,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "lines.linewidth": 1.8,
})

### Exact BBKS peak number density

$f(x)$ is eq. (A15), $G(\gamma, w)$ is eq. (A19), and the differential / cumulative densities follow eq. (4.3) with the exact $G$.

In [ ]:
def f_exact(x):
    """Exact f(x), BBKS eq. (A15), for x >= 0."""
    x = np.asarray(x, dtype=float)
    term1 = 0.5 * (x**3 - 3 * x) * (erf(np.sqrt(5 / 2) * x) + erf(np.sqrt(5 / 2) * x / 2))
    term2 = np.sqrt(2 / (5 * np.pi)) * ((31 * x**2 / 4 + 8 / 5) * np.exp(-5 * x**2 / 8)
                                        + (x**2 / 2 - 8 / 5) * np.exp(-5 * x**2 / 2))
    return term1 + term2


def G_exact(gamma, w, x_upper=60.0):
    """Exact G(gamma, w), eq. (A19): f_exact integrated against a Gaussian kernel."""
    sigma = np.sqrt(1.0 - gamma**2)

    def integrand(x):
        return f_exact(x) * np.exp(-(x - w)**2 / (2 * sigma**2)) / (np.sqrt(2 * np.pi) * sigma)

    upper = max(x_upper, w + 10 * sigma + 10.0)
    val, _ = quad(integrand, 0.0, upper, limit=300)
    return val


def exact_differential_number_density(nu, gamma, R_star):
    """N_pk(nu): comoving differential peak number density per unit nu (eq. 4.3, exact G)."""
    nu = np.asarray(nu, dtype=float)
    prefactor = 1.0 / (2.0 * np.pi)**2 / R_star**3
    flat = np.atleast_1d(nu)
    result = np.array([prefactor * np.exp(-n**2 / 2.0) * G_exact(gamma, gamma * n) for n in flat])
    return result.item() if nu.ndim == 0 else result


def exact_cumulative_number_density(nu, gamma, R_star, nu_upper=40.0):
    """n_pk(>nu): exact differential density integrated from nu to infinity."""
    nu = np.asarray(nu, dtype=float)
    flat = np.atleast_1d(nu)
    result = np.empty_like(flat)
    for i, nu_i in enumerate(flat):
        upper = max(nu_upper, nu_i + 10.0)
        result[i], _ = quad(lambda x: exact_differential_number_density(x, gamma, R_star),
                            nu_i, upper, limit=100)
    return result.item() if nu.ndim == 0 else result

### CSV readers

In [ ]:
def iter_profiles_csv(path):
    """Yield one profile y at a time. The first non-empty row (the u grid) is skipped."""
    with open(path, newline="") as f:
        reader = csv.reader(f)
        for row in reader:          # skip the u grid
            if row:
                break
        for row in reader:
            if row:
                yield np.array(row, dtype=float)

In [ ]:
# extract number density of peaks

csv_path_ga  = "peaks_0.csv"
csv_path_ng1 = "peaks_2.csv"
csv_path_ng2 = "peaks_-2.csv"

nu = np.linspace(2.5, 6.0, 50)
dnu = nu[1] - nu[0]

L = 40
N_rea = 5000
PROFILES_PER_PEAK = 6          # each peak appears 6 times in the files (profile extraction)
gamma_file = 0.77459667
Rs_file = 1.0954451


def number_density_from_csv(path):
    """Histogram of peak heights y[0] on the nu grid (bins centred on nu, width dnu),
    normalised to a comoving number density per unit nu."""
    counts = np.zeros_like(nu)
    for y in iter_profiles_csv(path):
        i = int(np.floor((y[0] - nu[0]) / dnu + 0.5))   # bin: [nu_i - dnu/2, nu_i + dnu/2)
        if 0 <= i < len(nu):
            counts[i] += 1
    return counts / (PROFILES_PER_PEAK * dnu * L**3 * N_rea)


number_density_ga  = number_density_from_csv(csv_path_ga)
number_density_ng1 = number_density_from_csv(csv_path_ng1)
number_density_ng2 = number_density_from_csv(csv_path_ng2)

In [ ]:
def plot_number_density(logy=False):
    plt.plot(nu, number_density_ga,  "o-", label=r"$\beta_{\rm NG} = 0$")
    plt.plot(nu, number_density_ng1, "s-", label=r"$\beta_{\rm NG} = 2$")
    plt.plot(nu, number_density_ng2, "d-", label=r"$\beta_{\rm NG} = -2$")
    plt.plot(nu, exact_differential_number_density(nu, gamma_file, Rs_file), "--",
             label=r"BBKS $\mathcal{N}_{\rm pk}(\nu)$")
    plt.xlabel(r"$\nu$", fontsize=12)
    plt.ylabel(r"$\mathcal{N}_{\rm pk}$", fontsize=12)
    if logy:
        plt.yscale("log")
        plt.ylim(1e-6, 1e-2)
    plt.legend(fontsize=10)
    plt.show()


plot_number_density()

In [ ]:
plot_number_density(logy=True)

### Expected number of peaks over a threshold from a single realization

In [ ]:
NU_TEST = 2.4
BOX_SIZE = 20.0

expected_peaks = exact_cumulative_number_density(NU_TEST, gamma_file, Rs_file) * BOX_SIZE**3
print(f"n_pk(>nu={NU_TEST:g}) * L^3 = {expected_peaks:.6g}  for L={BOX_SIZE:g}")